# Module 5: RAG Pipeline with Azure DocumentDB (C# completed reference)

In [ ]:
#r "nuget: MongoDB.Driver, 3.4.0"using MongoDB.Bson;using MongoDB.Driver;using System.Linq;var cs=Environment.GetEnvironmentVariable("DOCUMENTDB_CONNECTION_STRING");if(string.IsNullOrWhiteSpace(cs)) throw new Exception("Set DOCUMENTDB_CONNECTION_STRING");var client=new MongoClient(cs);var db=client.GetDatabase("docdbworkshop");var collection=db.GetCollection<BsonDocument>("rag_chunks");db.RunCommand<BsonDocument>(new BsonDocument("ping",1))

In [ ]:
collection.DeleteMany(FilterDefinition<BsonDocument>.Empty);collection.InsertMany(new[]{ new BsonDocument{{"_id","rag-001"},{"title","Vector search"},{"chunk","Azure DocumentDB vector search uses the $search stage with the cosmosSearch operator to retrieve documents by embedding similarity."},{"url","module-4-search"},{"embedding",new BsonArray{0.92,0.80,0.18}}}, new BsonDocument{{"_id","rag-002"},{"title","Full-text search"},{"chunk","Azure DocumentDB full-text search uses createSearchIndexes and the $search text operator to return BM25-ranked keyword matches."},{"url","module-4-search"},{"embedding",new BsonArray{0.20,0.12,0.94}}}, new BsonDocument{{"_id","rag-003"},{"title","Hybrid search"},{"chunk","Hybrid search runs BM25 keyword retrieval and vector retrieval, then combines ranked lists with Reciprocal Rank Fusion."},{"url","module-4-search"},{"embedding",new BsonArray{0.76,0.70,0.42}}}, new BsonDocument{{"_id","rag-004"},{"title","Grounded generation"},{"chunk","A RAG pipeline retrieves relevant chunks from Azure DocumentDB and includes them in the model prompt so the answer is grounded in current application data."},{"url","module-5-rag"},{"embedding",new BsonArray{0.84,0.73,0.34}}}});

In [ ]:
db.RunCommand<BsonDocument>(new BsonDocument{{"createIndexes","rag_chunks"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_chunk_embedding_diskann"},{"key",new BsonDocument("embedding","cosmosSearch")},{"cosmosSearchOptions",new BsonDocument{{"kind","vector-diskann"},{"dimensions",3},{"similarity","COS"},{"maxDegree",32},{"lBuild",64}}}}}}});db.RunCommand<BsonDocument>(new BsonDocument{{"createSearchIndexes","rag_chunks"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_chunk_fts"},{"definition",new BsonDocument("mappings",new BsonDocument{{"dynamic",false},{"fields",new BsonDocument("chunk",new BsonDocument("type","string"))}})}}}}});

## Retrieve context

In [ ]:
var question="How does DocumentDB retrieve context for RAG?"; var qv=new BsonArray{0.83,0.74,0.33};var vectorContext=collection.Aggregate<BsonDocument>(new[]{new BsonDocument("$search",new BsonDocument("cosmosSearch",new BsonDocument{{"path","embedding"},{"vector",qv},{"k",3}})),new BsonDocument("$project",new BsonDocument{{"_id",1},{"title",1},{"chunk",1},{"url",1},{"score",new BsonDocument("$meta","searchScore")}})}).ToList();vectorContext

## Build grounded prompt

In [ ]:
var contextBlock=string.Join("\n\n",vectorContext.Select((d,i)=>$"[{i+1}] {d["title"]}\n{d["chunk"]}\nSource: {d["url"]}"));var groundedPrompt=$"""You are a helpful assistant for an Azure DocumentDB workshop.Answer using only the context below. If the answer is missing, say you do not know.<context>{contextBlock}</context>Question: How does DocumentDB retrieve context for RAG?""";groundedPrompt